# QoS Sentry — DSO2 : Detection d'Anomalies Reseau (Non-Supervise)
### Isolation Forest | One-Class SVM | Autoencoder | LSTM AE | BiLSTM AE | GRU AE

**Approche NVIDIA** : Les autoencoders sont entraines **uniquement sur les donnees NORMALES**.
Ils apprennent a reconstruire le trafic normal — tout ce qui est difficile a reconstruire = anomalie.

**Threshold** : Seuil de Youden — maximise `TPR - FPR` sur la courbe ROC.

**Split** :
- Train : donnees normales uniquement (80%)
- Validation : donnees normales uniquement (20%)
- Test : toutes les donnees (normales + anomalies)


## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, os

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    f1_score, roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.decomposition import PCA

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    TORCH  = True
    torch.manual_seed(42)
    print(f'PyTorch {torch.__version__} | device={DEVICE}')
except ImportError:
    TORCH = False
    print('PyTorch non installe. Autoencoder simple (sklearn MLP) sera utilise.')
    print('Pour activer LSTM/BiLSTM/GRU : pip install torch')

warnings.filterwarnings('ignore')
np.random.seed(42)

# Style fond blanc
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': '#d1d5db', 'grid.color': '#e5e7eb',
    'grid.linestyle': '--', 'grid.alpha': 0.7,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
})

DATA_PATH = '/mnt/user-data/uploads/network_qoe_latestin.csv'
OUT_DIR   = '/mnt/user-data/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

# Hyperparametres
SEQ_LEN = 16
HIDDEN  = 64
LATENT  = 16
LR      = 1e-3
EPOCHS  = 100
PATIENCE= 12
BATCH   = 256

COLORS  = ['#2563eb','#dc2626','#16a34a','#d97706','#7c3aed','#0891b2']
print('Configuration OK')


## 1. Donnees — Apercu

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f'Lignes      : {len(df):,}')
print(f'Colonnes    : {df.shape[1]}')
print(f'Manquantes  : {df.isnull().sum().sum()}')
print()
print('Types de donnees :')
display(df.dtypes.to_frame('Type').T)
print()
print('Distribution des labels :')
vc = df['label'].value_counts()
for l,n in vc.items():
    print(f'  {l:<25} {n:>6,}  ({n/len(df)*100:.1f}%)')
print(f'\nTaux anomalie : {(df["label"]!="NORMAL").mean()*100:.1f}%')
display(df.describe().round(3))


## 2. EDA

In [ ]:
vc     = df['label'].value_counts()
colors = COLORS[:len(vc)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution des classes (reference — non utilisees a l entrainement)')

axes[0].barh(vc.index, vc.values, color=colors, edgecolor='white', height=0.65)
for i,(v,l) in enumerate(zip(vc.values, vc.index)):
    axes[0].text(v+200, i, f'{v:,}  ({v/len(df)*100:.1f}%)', va='center', fontsize=10)
axes[0].set_xlabel("Nombre d'echantillons")
axes[0].set_title('Comptes par classe'); axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.4)

axes[1].pie(vc.values, labels=vc.index, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2))
axes[1].set_title('Proportions')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/eda_01_labels.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Distribution des scores de reconstruction selon le label (approche NVIDIA)
# Montrer que les metriques QoE separent bien normal vs anomalie
KEY_FEATS = ['e2e_delay_ms','plr','jitter_ms','mos_voice',
             'throughput_mbps','buffering_ratio']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Distributions des metriques QoS par scenario')
l2c = dict(zip(df['label'].unique(), COLORS))

for ax, feat in zip(axes.flat, KEY_FEATS):
    for lbl, col in l2c.items():
        s = df[df['label']==lbl][feat].dropna()
        s = s[s <= s.quantile(0.99)]
        try: s.plot.kde(ax=ax, label=lbl, color=col, lw=2, alpha=0.85)
        except: pass
    ax.set_title(feat); ax.set_xlabel('')
    ax.legend(fontsize=7, framealpha=0.7); ax.grid(alpha=0.4)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/eda_02_kde.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Matrice de correlation
qoe = ['e2e_delay_ms','plr','jitter_ms','mos_voice','throughput_mbps',
       'buffering_ratio','availability','dns_latency_ms','streaming_mos',
       'effective_bitrate_mbps','call_setup_time_ms']

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(df[qoe].corr(), dtype=bool))
sns.heatmap(df[qoe].corr(), mask=mask,
            cmap=sns.diverging_palette(230, 20, as_cmap=True),
            center=0, annot=True, fmt='.2f', annot_kws={'size':8},
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'shrink':0.8})
ax.set_title('Matrice de correlation — metriques QoE')
for t in ax.get_xticklabels(): t.set_rotation(30); t.set_fontsize(9)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/eda_03_correlation.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Preprocessing & Split

> **Approche NVIDIA** : Entrainer les autoencoders UNIQUEMENT sur les donnees normales.  
> Le modele apprend la distribution du trafic normal.  
> A l'inference : erreur de reconstruction elevee = anomalie.


In [ ]:
FEATURES = [
    'e2e_delay_ms', 'plr', 'jitter_ms', 'mos_voice', 'throughput_mbps',
    'buffering_ratio', 'availability', 'call_setup_time_ms',
    'streaming_mos', 'dns_latency_ms', 'cdr_flag',
    'effective_bitrate_mbps', 'ctrl_plane_rtt_ms', 'flow_count',
]

# Labels ground truth (UNIQUEMENT pour evaluation finale)
y_all = (df['label'] != 'NORMAL').astype(int).values

X_all = df[FEATURES].fillna(0).values

# Scaler entraine sur les donnees normales uniquement
X_normal = X_all[y_all == 0]  # donnees normales
X_anomal = X_all[y_all == 1]  # donnees anormales

scaler = MinMaxScaler()
scaler.fit(X_normal)  # fit sur NORMAL uniquement
X_scaled = scaler.transform(X_all)
X_norm_s = scaler.transform(X_normal)

# Split des donnees normales : 80% train, 20% validation
X_train, X_val = train_test_split(X_norm_s, test_size=0.20, random_state=42)

# Test set : TOUTES les donnees (normal + anomalie)
X_test = X_scaled
y_test = y_all

print(f'Features         : {len(FEATURES)}')
print(f'Train (normal)   : {X_train.shape[0]:,}')
print(f'Val   (normal)   : {X_val.shape[0]:,}')
print(f'Test  (all)      : {X_test.shape[0]:,}')
print(f'  dont normal    : {(y_test==0).sum():,}')
print(f'  dont anomalie  : {(y_test==1).sum():,}')


In [ ]:
pca2  = PCA(n_components=2, random_state=42)
X_2d  = pca2.fit_transform(X_scaled)
labels_unique = list(df['label'].unique())

fig, ax = plt.subplots(figsize=(10, 7))
for i, lbl in enumerate(labels_unique):
    m = df['label'].values == lbl
    ax.scatter(X_2d[m,0], X_2d[m,1], c=COLORS[i%6],
               label=lbl, s=4, alpha=0.4, linewidths=0)
ax.set_title(f'PCA 2D — variance expliquee {sum(pca2.explained_variance_ratio_)*100:.1f}%')
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
ax.legend(markerscale=3, fontsize=9, framealpha=0.8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/eda_04_pca.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Fonctions d'Evaluation

In [ ]:
def youden_threshold(y_true, scores):
    """
    Seuil optimal de Youden : maximise TPR - FPR sur la courbe ROC.
    Approche utilisee dans le notebook NVIDIA.
    """
    fpr_v, tpr_v, thresholds = roc_curve(y_true, scores)
    J       = tpr_v - fpr_v
    best_i  = np.argmax(J)
    return float(thresholds[best_i])


def evaluate(name, y_true, scores, threshold):
    """Calcule toutes les metriques de classification."""
    preds = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return {
        'name':       name,
        'threshold':  round(threshold, 4),
        'Accuracy':   accuracy_score(y_true, preds),
        'Recall':     recall_score(y_true, preds, zero_division=0),
        'Precision':  precision_score(y_true, preds, zero_division=0),
        'F1':         f1_score(y_true, preds, zero_division=0),
        'FPR':        fp / (fp+tn) if (fp+tn)>0 else 0.0,
        'AUC':        roc_auc_score(y_true, scores),
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
        'scores': scores,
        'preds':  preds,
    }


def plot_cm(r, ax):
    """Matrice de confusion avec metriques."""
    cm = np.array([[r['TN'], r['FP']], [r['FN'], r['TP']]])
    im = ax.imshow(cm, cmap='Blues', aspect='auto')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Normal', 'Anomalie'])
    ax.set_yticklabels(['Normal', 'Anomalie'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black',
                    fontsize=10, fontweight='bold')
    ax.set_xlabel('Predit'); ax.set_ylabel('Reel')
    ax.set_title(
        f"{r['name']}\n"
        f"Acc={r['Accuracy']*100:.1f}%  F1={r['F1']:.3f}  AUC={r['AUC']:.3f}"
    )
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)


def plot_recon_dist(ax, scores, y_true, threshold, title):
    """
    Distribution des scores de reconstruction (approche NVIDIA Section 3).
    Montre la separation entre normal et anomalie.
    """
    bins = np.linspace(0, 1, 80)
    ax.hist(scores[y_true==0], bins=bins, color='#2563eb', alpha=0.6,
            label='Normal', density=True)
    ax.hist(scores[y_true==1], bins=bins, color='#dc2626', alpha=0.6,
            label='Anomalie', density=True)
    ax.axvline(threshold, color='black', ls='--', lw=2,
               label=f'Seuil Youden={threshold:.3f}')
    ax.set_title(title); ax.set_xlabel('Score de reconstruction')
    ax.set_ylabel('Densite'); ax.legend(fontsize=9); ax.grid(alpha=0.4)


all_results = {}
print('Fonctions definies.')


## 5. Isolation Forest (Baseline)

In [ ]:
# Entraine sur donnees NORMALES uniquement
iso = IsolationForest(
    n_estimators=300,
    contamination='auto',
    random_state=42,
    n_jobs=-1
)
iso.fit(X_train)

# Score : plus eleve = plus anormal
scores_iso = -iso.decision_function(X_test)
scores_iso = (scores_iso - scores_iso.min()) / (scores_iso.max() - scores_iso.min() + 1e-9)

# Seuil de Youden
thr_iso = youden_threshold(y_test, scores_iso)
r_iso   = evaluate('Isolation Forest', y_test, scores_iso, thr_iso)
all_results['Isolation Forest'] = r_iso

print(f'Seuil Youden  : {thr_iso:.4f}')
print(f'Accuracy      : {r_iso["Accuracy"]*100:.2f}%')
print(f'Recall        : {r_iso["Recall"]*100:.2f}%')
print(f'Precision     : {r_iso["Precision"]:.4f}')
print(f'F1 Score      : {r_iso["F1"]:.4f}')
print(f'FPR           : {r_iso["FPR"]*100:.2f}%')
print(f'AUC-ROC       : {r_iso["AUC"]:.4f}')


## 6. One-Class SVM

In [ ]:
# Sous-echantillonnage pour la vitesse
N_SVM = min(8000, len(X_train))
idx_svm = np.random.choice(len(X_train), N_SVM, replace=False)

svm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
svm.fit(X_train[idx_svm])

scores_svm = -svm.decision_function(X_test)
scores_svm = (scores_svm - scores_svm.min()) / (scores_svm.max() - scores_svm.min() + 1e-9)

thr_svm = youden_threshold(y_test, scores_svm)
r_svm   = evaluate('One-Class SVM', y_test, scores_svm, thr_svm)
all_results['One-Class SVM'] = r_svm

print(f'Seuil Youden  : {thr_svm:.4f}')
print(f'Accuracy      : {r_svm["Accuracy"]*100:.2f}%')
print(f'Recall        : {r_svm["Recall"]*100:.2f}%')
print(f'Precision     : {r_svm["Precision"]:.4f}')
print(f'F1 Score      : {r_svm["F1"]:.4f}')
print(f'FPR           : {r_svm["FPR"]*100:.2f}%')
print(f'AUC-ROC       : {r_svm["AUC"]:.4f}')


## 7. Autoencoder (MLP Dense)

Architecture : `Input → 128 → 64 → 32 → 64 → 128 → Output`  
Entraine UNIQUEMENT sur les donnees normales (approche NVIDIA).


In [ ]:
from sklearn.neural_network import MLPRegressor

ae_mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32, 64, 128),
    activation='relu',
    solver='adam',
    learning_rate_init=LR,
    max_iter=EPOCHS,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=PATIENCE,
    verbose=False
)
ae_mlp.fit(X_train, X_train)  # X -> X reconstruction, normal uniquement

print(f'Early stopping @ iteration : {ae_mlp.n_iter_}')

# Erreur de reconstruction = score d'anomalie
X_recon_mlp  = ae_mlp.predict(X_test)
err_mlp      = np.mean((X_test - X_recon_mlp)**2, axis=1)
scores_ae    = (err_mlp - err_mlp.min()) / (err_mlp.max() - err_mlp.min() + 1e-9)

# Stats de reconstruction (approche NVIDIA Section 3)
print(f'\nStats de reconstruction :')
err_df = pd.DataFrame({'recon_score': scores_ae, 'is_anomaly': y_test})
display(err_df.groupby('is_anomaly')['recon_score'].describe().round(4))
print('=> Score moyen anomalie > Score moyen normal : modele discriminant')


In [ ]:
# Courbe de perte early stopping
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ae_mlp.loss_curve_, color='#2563eb', lw=2, label='Train loss')
if hasattr(ae_mlp, 'validation_scores_') and ae_mlp.validation_scores_:
    ax.plot([-v for v in ae_mlp.validation_scores_],
            color='#dc2626', lw=2, ls='--', label='Val loss')
ax.axvline(ae_mlp.n_iter_, color='#16a34a', ls=':', lw=1.5,
           label=f'Best @ iter {ae_mlp.n_iter_}')
ax.set_title('Autoencoder MLP — Courbe de perte'); ax.set_xlabel('Iteration')
ax.set_ylabel('MSE Loss'); ax.legend(); ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/ae_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

thr_ae = youden_threshold(y_test, scores_ae)
r_ae   = evaluate('Autoencoder', y_test, scores_ae, thr_ae)
all_results['Autoencoder'] = r_ae

print(f'Seuil Youden  : {thr_ae:.4f}')
print(f'Accuracy      : {r_ae["Accuracy"]*100:.2f}%')
print(f'Recall        : {r_ae["Recall"]*100:.2f}%')
print(f'Precision     : {r_ae["Precision"]:.4f}')
print(f'F1 Score      : {r_ae["F1"]:.4f}')
print(f'FPR           : {r_ae["FPR"]*100:.2f}%')
print(f'AUC-ROC       : {r_ae["AUC"]:.4f}')


## 8. Definition des Autoencoders PyTorch (LSTM / BiLSTM / GRU)

In [ ]:
if not TORCH:
    print('SKIP — PyTorch non disponible. Installer avec: pip install torch')
else:
    # ============================================================
    # LSTM Autoencoder
    # ============================================================
    class LSTMAutoencoder(nn.Module):
        def __init__(self, n_feat, hidden=HIDDEN, latent=LATENT):
            super().__init__()
            self.encoder = nn.LSTM(n_feat, hidden, batch_first=True)
            self.enc_fc  = nn.Sequential(nn.Linear(hidden, latent), nn.ReLU())
            self.decoder = nn.LSTM(latent, hidden, batch_first=True)
            self.dec_fc  = nn.Linear(hidden, n_feat)

        def forward(self, x):                  # x: (B, T, F)
            _, (h, _) = self.encoder(x)        # h: (1, B, H)
            z   = self.enc_fc(h.squeeze(0))    # (B, latent)
            z_r = z.unsqueeze(1).expand(-1, x.size(1), -1)
            out, _ = self.decoder(z_r)         # (B, T, H)
            return self.dec_fc(out)             # (B, T, F)

    # ============================================================
    # BiLSTM Autoencoder
    # ============================================================
    class BiLSTMAutoencoder(nn.Module):
        def __init__(self, n_feat, hidden=HIDDEN, latent=LATENT):
            super().__init__()
            self.encoder = nn.LSTM(n_feat, hidden, batch_first=True, bidirectional=True)
            self.enc_fc  = nn.Sequential(nn.Linear(2*hidden, latent), nn.ReLU())
            self.decoder = nn.LSTM(latent, hidden, batch_first=True)
            self.dec_fc  = nn.Linear(hidden, n_feat)

        def forward(self, x):
            _, (h, _) = self.encoder(x)                        # h: (2, B, H)
            h_cat = torch.cat([h[0], h[1]], dim=1)             # (B, 2H)
            z   = self.enc_fc(h_cat)                           # (B, latent)
            z_r = z.unsqueeze(1).expand(-1, x.size(1), -1)
            out, _ = self.decoder(z_r)
            return self.dec_fc(out)

    # ============================================================
    # GRU Autoencoder
    # ============================================================
    class GRUAutoencoder(nn.Module):
        def __init__(self, n_feat, hidden=HIDDEN, latent=LATENT):
            super().__init__()
            self.encoder = nn.GRU(n_feat, hidden, batch_first=True)
            self.enc_fc  = nn.Sequential(nn.Linear(hidden, latent), nn.ReLU())
            self.decoder = nn.GRU(latent, hidden, batch_first=True)
            self.dec_fc  = nn.Linear(hidden, n_feat)

        def forward(self, x):
            _, h = self.encoder(x)              # h: (1, B, H)
            z   = self.enc_fc(h.squeeze(0))
            z_r = z.unsqueeze(1).expand(-1, x.size(1), -1)
            out, _ = self.decoder(z_r)
            return self.dec_fc(out)

    print('Classes definies : LSTMAutoencoder | BiLSTMAutoencoder | GRUAutoencoder')


## 9. Entrainement avec Early Stopping (PyTorch)

In [ ]:
def make_sequences(X, seq_len, stride=None):
    """Decoupage en sequences glissantes."""
    if stride is None: stride = seq_len // 2
    seqs   = [X[i:i+seq_len] for i in range(0, len(X)-seq_len+1, stride)]
    starts = [i for i in range(0, len(X)-seq_len+1, stride)]
    return np.array(seqs), np.array(starts)


def train_ae(model, X_train, X_val, seq_len=SEQ_LEN,
             epochs=EPOCHS, lr=LR, batch=BATCH, patience=PATIENCE):
    """
    Entraine un autoencoder PyTorch sur donnees normales avec early stopping.
    Retourne (model, train_losses, val_losses).
    """
    model   = model.to(DEVICE)
    opt     = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    seqs_tr, _ = make_sequences(X_train, seq_len)
    seqs_va, _ = make_sequences(X_val,   seq_len)

    Xt = torch.FloatTensor(seqs_tr).to(DEVICE)
    Xv = torch.FloatTensor(seqs_va).to(DEVICE)
    loader = DataLoader(TensorDataset(Xt, Xt), batch_size=batch, shuffle=True)

    best_loss, best_state, patience_cnt = float('inf'), None, 0
    tr_hist, vl_hist = [], []

    for ep in range(1, epochs+1):
        model.train(); ep_loss = 0.0
        for xb, _ in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), xb)
            loss.backward(); opt.step()
            ep_loss += loss.item()
        ep_loss /= len(loader)

        model.eval()
        with torch.no_grad():
            vl_loss = loss_fn(model(Xv), Xv).item()

        tr_hist.append(ep_loss); vl_hist.append(vl_loss)

        if vl_loss < best_loss - 1e-6:
            best_loss      = vl_loss
            best_state     = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt   = 0
        else:
            patience_cnt  += 1

        if ep % 10 == 0:
            print(f'  Epoch {ep:3d}/{epochs} | train={ep_loss:.5f} | val={vl_loss:.5f} | patience={patience_cnt}/{patience}')

        if patience_cnt >= patience:
            print(f'  => Early stopping @ epoch {ep} | best val={best_loss:.5f}')
            break

    if best_state:
        model.load_state_dict(best_state)
    return model.to('cpu'), tr_hist, vl_hist


def recon_scores(model, X, seq_len=SEQ_LEN):
    """
    Calcule les scores de reconstruction pour chaque sample.
    Score = moyenne des MSE des sequences contenant ce sample.
    """
    stride = seq_len // 2
    seqs, starts = make_sequences(X, seq_len, stride)
    model.eval()
    err_sum = np.zeros(len(X)); err_cnt = np.zeros(len(X))

    with torch.no_grad():
        for b in range(0, len(seqs), 512):
            batch  = torch.FloatTensor(seqs[b:b+512])
            recon  = model(batch).numpy()
            mse    = np.mean((seqs[b:b+512] - recon)**2, axis=(1, 2))
            for k, err in enumerate(mse):
                s = starts[b+k]; e = min(s+seq_len, len(X))
                err_sum[s:e] += err; err_cnt[s:e] += 1

    err_cnt = np.where(err_cnt == 0, 1, err_cnt)
    raw     = err_sum / err_cnt
    return (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)


if TORCH:
    print('Fonctions train_ae et recon_scores definies.')
else:
    print('PyTorch absent — cellules LSTM/BiLSTM/GRU seront ignorees.')


## 10. LSTM Autoencoder

In [ ]:
if TORCH:
    print('=== LSTM Autoencoder ===')
    lstm_m = LSTMAutoencoder(n_feat=len(FEATURES))
    lstm_m, lstm_tr, lstm_vl = train_ae(lstm_m, X_train, X_val)

    sc_lstm = recon_scores(lstm_m, X_test)
    thr_l   = youden_threshold(y_test, sc_lstm)
    r_lstm  = evaluate('LSTM AE', y_test, sc_lstm, thr_l)
    all_results['LSTM AE'] = r_lstm

    print(f'Seuil Youden  : {thr_l:.4f}')
    print(f'Accuracy      : {r_lstm["Accuracy"]*100:.2f}%')
    print(f'Recall        : {r_lstm["Recall"]*100:.2f}%')
    print(f'Precision     : {r_lstm["Precision"]:.4f}')
    print(f'F1 Score      : {r_lstm["F1"]:.4f}')
    print(f'FPR           : {r_lstm["FPR"]*100:.2f}%')
    print(f'AUC-ROC       : {r_lstm["AUC"]:.4f}')
else:
    print('SKIP — PyTorch requis')


## 11. BiLSTM Autoencoder

In [ ]:
if TORCH:
    print('=== BiLSTM Autoencoder ===')
    bilstm_m = BiLSTMAutoencoder(n_feat=len(FEATURES))
    bilstm_m, bilstm_tr, bilstm_vl = train_ae(bilstm_m, X_train, X_val)

    sc_bilstm = recon_scores(bilstm_m, X_test)
    thr_b     = youden_threshold(y_test, sc_bilstm)
    r_bilstm  = evaluate('BiLSTM AE', y_test, sc_bilstm, thr_b)
    all_results['BiLSTM AE'] = r_bilstm

    print(f'Seuil Youden  : {thr_b:.4f}')
    print(f'Accuracy      : {r_bilstm["Accuracy"]*100:.2f}%')
    print(f'Recall        : {r_bilstm["Recall"]*100:.2f}%')
    print(f'Precision     : {r_bilstm["Precision"]:.4f}')
    print(f'F1 Score      : {r_bilstm["F1"]:.4f}')
    print(f'FPR           : {r_bilstm["FPR"]*100:.2f}%')
    print(f'AUC-ROC       : {r_bilstm["AUC"]:.4f}')
else:
    print('SKIP — PyTorch requis')


## 12. GRU Autoencoder

In [ ]:
if TORCH:
    print('=== GRU Autoencoder ===')
    gru_m = GRUAutoencoder(n_feat=len(FEATURES))
    gru_m, gru_tr, gru_vl = train_ae(gru_m, X_train, X_val)

    sc_gru = recon_scores(gru_m, X_test)
    thr_g  = youden_threshold(y_test, sc_gru)
    r_gru  = evaluate('GRU AE', y_test, sc_gru, thr_g)
    all_results['GRU AE'] = r_gru

    print(f'Seuil Youden  : {thr_g:.4f}')
    print(f'Accuracy      : {r_gru["Accuracy"]*100:.2f}%')
    print(f'Recall        : {r_gru["Recall"]*100:.2f}%')
    print(f'Precision     : {r_gru["Precision"]:.4f}')
    print(f'F1 Score      : {r_gru["F1"]:.4f}')
    print(f'FPR           : {r_gru["FPR"]*100:.2f}%')
    print(f'AUC-ROC       : {r_gru["AUC"]:.4f}')
else:
    print('SKIP — PyTorch requis')


## 13. Modele Ensemble — GRU AE + BiLSTM AE + Isolation Forest

**Principe** : Combiner trois modeles complementaires par moyenne ponderee des scores.  
Les poids sont proportionnels a l'AUC individuel de chaque modele.  
- **GRU AE** : capture les patterns temporels locaux  
- **BiLSTM AE** : capture les dependances temporelles passees ET futures  
- **Isolation Forest** : detection par isolation statistique (complementaire aux AE)


In [ ]:
if TORCH:
    print('=== Modele Ensemble (GRU AE + BiLSTM AE + Isolation Forest) ===')

    # Verifier que les 3 modeles sont disponibles
    required = ['GRU AE', 'BiLSTM AE', 'Isolation Forest']
    missing  = [m for m in required if m not in all_results]
    if missing:
        print(f'Modeles manquants : {missing} — executer les cellules precedentes d abord')
    else:
        # Recuperer les scores normalises de chaque modele
        sc_gru_e  = all_results['GRU AE']['scores']
        sc_bi_e   = all_results['BiLSTM AE']['scores']
        sc_iso_e  = all_results['Isolation Forest']['scores']

        # Poids proportionnels a l'AUC (les meilleurs modeles contribuent plus)
        auc_gru   = all_results['GRU AE']['AUC']
        auc_bi    = all_results['BiLSTM AE']['AUC']
        auc_iso   = all_results['Isolation Forest']['AUC']
        total_auc = auc_gru + auc_bi + auc_iso

        w_gru  = auc_gru  / total_auc
        w_bi   = auc_bi   / total_auc
        w_iso  = auc_iso  / total_auc

        print(f'Poids AUC :')
        print(f'  GRU AE           : {w_gru:.3f}  (AUC={auc_gru:.4f})')
        print(f'  BiLSTM AE        : {w_bi:.3f}  (AUC={auc_bi:.4f})')
        print(f'  Isolation Forest : {w_iso:.3f}  (AUC={auc_iso:.4f})')

        # Score ensemble = moyenne ponderee
        sc_ens = w_gru * sc_gru_e + w_bi * sc_bi_e + w_iso * sc_iso_e
        sc_ens = (sc_ens - sc_ens.min()) / (sc_ens.max() - sc_ens.min() + 1e-9)

        # Seuil de Youden sur le score ensemble
        thr_ens = youden_threshold(y_test, sc_ens)
        r_ens   = evaluate('Ensemble', y_test, sc_ens, thr_ens)
        all_results['Ensemble'] = r_ens

        print(f'\nSeuil Youden  : {thr_ens:.4f}')
        print(f'Accuracy      : {r_ens["Accuracy"]*100:.2f}%')
        print(f'Recall        : {r_ens["Recall"]*100:.2f}%')
        print(f'Precision     : {r_ens["Precision"]:.4f}')
        print(f'F1 Score      : {r_ens["F1"]:.4f}')
        print(f'FPR           : {r_ens["FPR"]*100:.2f}%')
        print(f'AUC-ROC       : {r_ens["AUC"]:.4f}')
else:
    print('SKIP — PyTorch requis pour GRU AE et BiLSTM AE')


## 14. Evaluation Complete
### 14.1 Courbes Early Stopping

In [ ]:
if TORCH and len(all_results) >= 4:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Courbes de perte — Early Stopping (entrainement sur normaux uniquement)')

    for ax, (tr, vl, title) in zip(axes, [
        (lstm_tr,   lstm_vl,   'LSTM Autoencoder'),
        (bilstm_tr, bilstm_vl, 'BiLSTM Autoencoder'),
        (gru_tr,    gru_vl,    'GRU Autoencoder'),
    ]):
        ep = range(1, len(tr)+1)
        ax.plot(ep, tr, color='#2563eb', lw=2, label='Train')
        ax.plot(ep, vl, color='#dc2626', lw=2, ls='--', label='Validation')
        best = int(np.argmin(vl))+1
        ax.axvline(best, color='#16a34a', ls=':', lw=1.5, label=f'Best={best}')
        ax.set_title(title); ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE Loss'); ax.legend(fontsize=9); ax.grid(alpha=0.4)

    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/eval_01_early_stopping.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('PyTorch absent — graphe non disponible')


### 14.2 Distributions des scores de reconstruction (approche NVIDIA)

In [ ]:
n_models = len(all_results)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distributions des scores de reconstruction — Normal vs Anomalie')

for ax, (name, r) in zip(axes.flat, all_results.items()):
    plot_recon_dist(ax, r['scores'], y_test, r['threshold'], name)

# Cacher les axes vides
for ax in axes.flat[n_models:]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/eval_02_recon_dist.png', dpi=150, bbox_inches='tight')
plt.show()


### 14.3 Matrices de Confusion

In [ ]:
n  = len(all_results)
nc = min(n, 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Matrices de confusion — tous les modeles (jeu de test)')

for ax, (name, r) in zip(axes.flat, all_results.items()):
    plot_cm(r, ax)

for ax in axes.flat[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/eval_03_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


### 14.4 Courbes ROC

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
colors_roc = COLORS[:len(all_results)]

for (name, r), col in zip(all_results.items(), colors_roc):
    fpr_v, tpr_v, _ = roc_curve(y_test, r['scores'])
    lw = 3 if 'Ensemble' in name else 1.8
    ls = '-'
    ax.plot(fpr_v, tpr_v, color=col, lw=lw, ls=ls,
            label=f"{name} (AUC={r['AUC']:.3f})")

ax.plot([0,1],[0,1], color='#9ca3af', ls=':', lw=1.5, label='Aleatoire')
ax.fill_between([0,1],[0,1], alpha=0.04, color='#9ca3af')
ax.set_xlim([-0.02,1.02]); ax.set_ylim([-0.02,1.02])
ax.set_xlabel('Taux Faux Positifs (FPR)', fontsize=12)
ax.set_ylabel('Recall (TPR)', fontsize=12)
ax.set_title('Courbes ROC — Comparaison des modeles', fontsize=13)
ax.legend(fontsize=11, framealpha=0.9, loc='lower right')
ax.grid(alpha=0.35)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/eval_04_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()


### 14.5 Tableau de Comparaison des Modeles

In [ ]:
comp = pd.DataFrame([{
    'Modele':        r['name'],
    'Accuracy (%)':  round(r['Accuracy']  * 100, 2),
    'Recall (%)':    round(r['Recall']    * 100, 2),
    'Precision':     round(r['Precision'],        4),
    'F1 Score':      round(r['F1'],               4),
    'FPR (%)':       round(r['FPR']       * 100, 2),
    'AUC-ROC':       round(r['AUC'],              4),
    'TP': r['TP'], 'FP': r['FP'], 'FN': r['FN'], 'TN': r['TN'],
    'Seuil':         r['threshold'],
} for r in all_results.values()])

comp = comp.sort_values('AUC-ROC', ascending=False).reset_index(drop=True)

display(
    comp.set_index('Modele').style
    .background_gradient(subset=['Accuracy (%)','Recall (%)','F1 Score','AUC-ROC'], cmap='Greens')
    .background_gradient(subset=['FPR (%)'], cmap='Reds_r')
    .format({
        'Accuracy (%)': '{:.2f}', 'Recall (%)': '{:.2f}',
        'Precision': '{:.4f}', 'F1 Score': '{:.4f}',
        'FPR (%)': '{:.2f}', 'AUC-ROC': '{:.4f}', 'Seuil': '{:.4f}'
    })
)

comp.to_csv(f'{OUT_DIR}/model_comparison.csv', index=False)
best = comp.iloc[0]
print(f'Meilleur modele (AUC) : {best["Modele"]}')
print(f'  Accuracy  = {best["Accuracy (%)"]:.2f}%')
print(f'  Recall    = {best["Recall (%)"]:.2f}%')
print(f'  Precision = {best["Precision"]:.4f}')
print(f'  F1 Score  = {best["F1 Score"]:.4f}')
print(f'  FPR       = {best["FPR (%)"]:.2f}%')
print(f'  AUC-ROC   = {best["AUC-ROC"]:.4f}')


### 14.6 Graphiques Comparatifs

In [ ]:
names = comp['Modele'].tolist()
clrs  = COLORS[:len(names)]
x     = np.arange(len(names))

fig, axes = plt.subplots(2, 3, figsize=(20, 11))
fig.suptitle('Comparaison des modeles — metriques cles (jeu de test, seuil Youden)')

metrics_cfg = [
    ('Accuracy (%)',  'Accuracy (%)'),
    ('Recall (%)',    'Recall (%)'),
    ('Precision',     'Precision'),
    ('F1 Score',      'F1 Score'),
    ('FPR (%)',       'FPR (%)'),
    ('AUC-ROC',       'AUC-ROC'),
]

for ax, (col, title) in zip(axes.flat, metrics_cfg):
    vals = [comp.loc[comp['Modele']==n, col].values[0] for n in names]
    bars = ax.bar(x, vals, color=clrs, width=0.6, edgecolor='white', alpha=0.88)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height() + max(vals)*0.01,
                f'{val:.3f}', ha='center', va='bottom',
                fontsize=9, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
    ax.set_title(title); ax.grid(axis='y', alpha=0.4)
    if col == 'FPR (%)':
        ax.set_facecolor('#fff5f5')  # fond rouge clair pour FPR (plus bas = mieux)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/eval_05_bar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 15. Resume Final

In [ ]:
print('='*70)
print('QoS SENTRY — DSO2 — RESUME FINAL')
print('='*70)
print('Approche  : Non-supervisee (entrainement sur donnees NORMALES uniquement)')
print('Threshold : Seuil de Youden — maximise TPR - FPR sur ROC')
print(f'Backend   : {"PyTorch " + torch.__version__ if TORCH else "sklearn MLP"}')
print(f'Test set  : {len(y_test):,} samples ({y_test.sum():,} anomalies)')
print()
print(f'{"Modele":<20} {"Acc%":>7} {"Rec%":>7} {"Prec":>7} {"F1":>7} {"AUC":>7}')
print('-'*60)
for _, r in all_results.items():
    print(f'{r["name"]:<20}'
          f' {r["Accuracy"]*100:>7.2f}'
          f' {r["Recall"]*100:>7.2f}'
          f' {r["Precision"]:>7.4f}'
          f' {r["F1"]:>7.4f}'
          f' {r["AUC"]:>7.4f}')
print()
print('Ameliorations vs version precedente :')
print('  1. Entrainement sur normaux UNIQUEMENT (approche NVIDIA)')
print('  2. Seuil de Youden (optimal ROC) au lieu du percentile fixe')
print('  3. Scaler ajuste sur normaux uniquement')
print('  4. Architecture HIDDEN=64, LATENT=16 (plus grande capacite)')
print('  5. Early stopping patience=12 (convergence plus stable)')
print()
print('Fichiers generes :')
for f in sorted(os.listdir(OUT_DIR)):
    if f.endswith(('.png','.csv','.ipynb')):
        sz = os.path.getsize(f'{OUT_DIR}/{f}')
        print(f'  {f:<50} {sz//1024:>5} KB')
